In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Getting the input data cleaned and formatted as an array: input matrix X

df = pd.read_csv('train_titanicml.csv')

y = df['Survived'].to_numpy()
y = y.reshape(891, 1)
y_train = y[:757]
y_validate = y[757:]

df = df.drop(columns=['PassengerId','Name', 'Ticket', 'Cabin', 'Survived'])
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Sex'] = df['Sex'].map({'female':1, 'male': 0}).astype(int)
df['Embarked'] = df['Embarked'].fillna('S')
df.insert(7, 'Embarked_C', (df['Embarked'] == 'C').astype(int))
df.insert(8, 'Embarked_S', (df['Embarked'] == 'S').astype(int))
df.insert(9, 'Embarked_Q', (df['Embarked'] == 'Q').astype(int))
df = df.drop(columns='Embarked')

ft = len(df.columns)

df_train = df[:757]
df_mean = df_train.mean()
df_std = df_train.std()
df_train = (df_train - df_mean) / df_std
X_train = df_train.to_numpy()

df_validate = df[757:]
df_validate = (df_validate - df_mean) / df_std
X_validate = df_validate.to_numpy()


In [ ]:
# Making from scratch the Neural Network

# Parameters with He initialization for ReLu and Xavier initialization for Sigmoid
h1 = np.sqrt((6)/ ft)
W1 = np.random.uniform(-h1, h1, (ft, 3))
b1 = np.zeros((1,3))
h2 = np.sqrt((6) / 3)
W2 = np.random.uniform(-h2, h2, (3, 3))
b2 = np.zeros((1,3))
x1 = np.sqrt((6)/(3+1))
W3 = np.random.uniform(-x1, x1, (3, 1))
b3 = np.zeros((1,1))

for epoch in range(8000):
  # Forward pass
  X1W1 = X_train @ W1
  Z1 = X1W1 + b1
  a1 = np.where(Z1>0, Z1, 0)

  a1W2 = a1 @ W2
  Z2 = a1W2 + b2
  a2 = np.where(Z2>0, Z2, 0)

  a2W3 = a2 @ W3
  Z3 = a2W3 + b3

  opred = 1 / (1 + np.exp(-Z3))
  opred = np.clip(opred, 1e-15, 1 - 1e-15)

  L = -np.mean((y_train * np.log(opred)) + (1 - y_train) * (np.log(1 - opred)))

  # Backward propagation
  dL = 1
  d_opred = dL * (1 / len(X_train)) * ((opred - y_train) / (opred * (1 - opred)))
  d_Z3 = d_opred * (opred *(1-opred))
  d_W3 = (a2.T) @ d_Z3
  d_b3 = np.sum(d_Z3, axis=0, keepdims=True)
  d_a2 = d_Z3 @ W3.T
  d_Z2 = d_a2 *(Z2 > 0)
  d_W2 = (a1.T) @ d_Z2
  d_b2 = np.sum(d_Z2, axis=0, keepdims=True)
  d_a1 = d_Z2 @ W2.T
  d_Z1 = d_a1 *(Z1 > 0)
  d_W1 = (X_train.T) @ d_Z1
  d_b1 = np.sum(d_Z1, axis=0, keepdims=True)

  # Updating parameters by learning rate

  W1 -= (0.05) * d_W1
  b1 -= (0.05) * d_b1
  W2 -= (0.05) * d_W2
  b2 -= (0.05) * d_b2
  W3 -= (0.05) * d_W3
  b3 -= (0.05) * d_b3

  if epoch % 250 == 0:
    print(f'Training loss: {L}')

    # Validation Loss
    X1W1 = X_validate @ W1
    Z1 = X1W1 + b1
    a1 = np.where(Z1>0, Z1, 0)

    a1W2 = a1 @ W2
    Z2 = a1W2 + b2
    a2 = np.where(Z2>0, Z2, 0)

    a2W3 = a2 @ W3
    Z3 = a2W3 + b3

    opred = 1 / (1 + np.exp(-Z3))
    opred = np.clip(opred, 1e-15, 1 - 1e-15)

    L = -np.mean((y_validate * np.log(opred)) + (1 - y_validate) * (np.log(1 - opred)))
    print(f'Validation loss: {L}')
    print()

In [ ]:
# TitanicML data test

df2 = pd.read_csv('test_titanicml.csv')

df2 = df2.drop(columns=['PassengerId','Name', 'Ticket', 'Cabin'])
df2['Age'] = df2['Age'].fillna(df2['Age'].median())
df2['Fare'] = df2['Fare'].fillna(df2['Fare'].median())
df2['Sex'] = df2['Sex'].map({'female':1, 'male': 0}).astype(int)
df2['Embarked'] = df2['Embarked'].fillna('S')
df2.insert(7, 'Embarked_C', (df2['Embarked'] == 'C').astype(int))
df2.insert(8, 'Embarked_S', (df2['Embarked'] == 'S').astype(int))
df2.insert(9, 'Embarked_Q', (df2['Embarked'] == 'Q').astype(int))
df2 = df2.drop(columns='Embarked')
df2 = (df2 - df_mean) / df_std
X_test = df2.to_numpy()

In [ ]:
# Forward pass
X1W1 = X_test @ W1
Z1 = X1W1 + b1
a1 = np.where(Z1>0, Z1, 0)

a1W2 = a1 @ W2
Z2 = a1W2 + b2
a2 = np.where(Z2>0, Z2, 0)

a2W3 = a2 @ W3
Z3 = a2W3 + b3

opred = 1 / (1 + np.exp(-Z3))

pred = (opred >= 0.5).astype(int)

In [ ]:
test_df = pd.read_csv('test_titanicml.csv')
submission = pd.DataFrame({'PassengerId' : test_df['PassengerId'], 'Survived' : pred.flatten()})
submission.to_csv('submission.csv', index=False)